# Lezione 12 – Pratica: IA responsabile, etica, bias e interpretabilità

Questo notebook è la pratica autonoma della Lezione 12 del corso Advanced Deep Learning ("deep learning avanzato"). Qui applichi, con il codice, le idee centrali sull'intelligenza artificiale (IA) responsabile che hai lavorato nella lezione in diretta: l'equità di un modello rispetto a diversi gruppi demografici e l'interpretabilità delle sue decisioni.

## Cosa farai in questa pratica

- Verifichi un modello fittizio di concessione di credito e misuri se tratta allo stesso modo due gruppi demografici, con il tasso di veri positivi (TPR)
- Costruisci una tecnica semplice di interpretabilità che stima quale variabile pesa di più nelle decisioni del modello

## Prerequisito

Aver seguito la Lezione 12 ("IA responsabile: etica, bias e interpretabilità") e avere chiari i concetti di bias algoritmico e interpretabilità. Se non l'hai ancora seguita, rivedi prima il capitolo 13 del manuale.

**Esegui ogni cella di codice in ordine: i due esercizi condividono variabili tra loro.**

## Esercizio 1: verifica un modello fittizio di concessione di credito

Immagina di lavorare con una banca che usa un modello di apprendimento automatico per decidere se concedere un prestito. Il team dati ti consegna un dataset sintetico con due gruppi demografici, A e B, e un modello di regressione logistica già addestrato (`LogisticRegression` di scikit-learn) che prevede se ogni richiedente ripagherà il prestito (etichetta 1) o non lo ripagherà (etichetta 0).

Lo scheletro qui sotto genera i dati — con una differenza di rappresentanza deliberata tra i due gruppi, così da avere qualcosa di reale da verificare —, addestra il modello e lascia il risultato in `X_test`, `y_test`, `grupo_test` e `y_pred`.

Completa la funzione `calcular_diferencia_equidad(y_real, y_predicho, grupo)`, che verifica il modello con la metrica di differenza di opportunità uguale (*equal opportunity difference*): calcola il TPR — la proporzione di persone che ripagano effettivamente il prestito (etichetta reale 1) e alle quali il modello predice anche 1 — separatamente in ciascun gruppo, e restituisce la differenza tra il gruppo con il TPR migliore e quello con il TPR peggiore. Più il valore è vicino a zero, più il modello è equo tra i gruppi.

**Suggerimenti:**

- Scorri i gruppi unici presenti in `grupo` con `np.unique(grupo)`
- Per ogni gruppo, usa una maschera booleana (`grupo == g`) per tenere solo le righe di quel gruppo
- All'interno di ogni gruppo, il TPR è la proporzione di successi del modello (`y_predicho == 1`) tra le persone la cui etichetta reale è 1
- La differenza finale è il TPR massimo meno il TPR minimo tra i gruppi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)
n_por_grupo = 1000

# Grupo A: grupo mayoritario, bien representado en el histórico
edad_a = rng.normal(40, 10, n_por_grupo)
ingresos_a = rng.normal(50, 15, n_por_grupo)
antiguedad_a = rng.normal(8, 4, n_por_grupo)
score_a = 0.04 * ingresos_a + 0.3 * antiguedad_a - 5
etiqueta_a = (score_a + rng.normal(0, 3, n_por_grupo) > 0).astype(int)

# Grupo B: grupo minoritario, peor representado en el histórico.
# El proceso causal es el mismo, pero los ingresos declarados son
# sistemáticamente más bajos y la etiqueta tiene más ruido de medición:
# así se simula un sesgo de representación en el origen de los datos.
edad_b = rng.normal(38, 10, n_por_grupo)
ingresos_b = rng.normal(38, 15, n_por_grupo)
antiguedad_b = rng.normal(6, 4, n_por_grupo)
score_b = 0.04 * ingresos_b + 0.3 * antiguedad_b - 5
etiqueta_b = (score_b + rng.normal(0, 5, n_por_grupo) > 0).astype(int)

df = pd.concat([
    pd.DataFrame({"edad": edad_a, "ingresos": ingresos_a, "antiguedad": antiguedad_a,
                  "grupo": "A", "etiqueta": etiqueta_a}),
    pd.DataFrame({"edad": edad_b, "ingresos": ingresos_b, "antiguedad": antiguedad_b,
                  "grupo": "B", "etiqueta": etiqueta_b}),
], ignore_index=True)

X = df[["edad", "ingresos", "antiguedad"]].values
y = df["etiqueta"].values
grupo = df["grupo"].values

X_train, X_test, y_train, y_test, grupo_train, grupo_test = train_test_split(
    X, y, grupo, test_size=0.3, random_state=42, stratify=grupo
)

# Modelo ya entrenado que vas a auditar
modelo = LogisticRegression()
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)

nombres_columnas = ["edad", "ingresos", "antiguedad"]

print("Filas de test:", len(X_test))
print("Grupos presentes:", np.unique(grupo_test))

In [ ]:
def calcular_diferencia_equidad(y_real, y_predicho, grupo):
    """
    Calcula la tasa de verdaderos positivos (TPR) de cada grupo demográfico
    y la diferencia entre el grupo con mejor y peor TPR (diferencia de
    oportunidad igual). Un valor cercano a cero indica que el modelo trata
    a ambos grupos con la misma eficacia a la hora de detectar verdaderos
    positivos.

    Parámetros
    ----------
    y_real : array con la etiqueta real (0/1)
    y_predicho : array con la etiqueta predicha por el modelo (0/1)
    grupo : array con el grupo demográfico de cada fila (por ejemplo, "A" o "B")

    Devuelve
    --------
    tprs : dict {grupo: TPR}
    diferencia : diferencia entre el TPR máximo y el TPR mínimo
    """
    grupos_unicos = np.unique(grupo)
    tprs = {}

    for g in grupos_unicos:
        mascara_grupo = grupo == g
        y_real_grupo = y_real[mascara_grupo]
        y_predicho_grupo = y_predicho[mascara_grupo]

        # TODO: calcula la tasa de verdaderos positivos (TPR) de este grupo:
        # de las filas donde y_real_grupo es igual a 1, qué proporción tiene
        # también y_predicho_grupo igual a 1
        tpr_grupo = ...

        tprs[g] = tpr_grupo

    # TODO: calcula la diferencia entre el TPR máximo y el TPR mínimo de tprs
    diferencia = ...

    return tprs, diferencia


tprs_por_grupo, diferencia = calcular_diferencia_equidad(y_test, y_pred, grupo_test)

print("TPR por grupo:")
for g, tpr in tprs_por_grupo.items():
    print(f"  Grupo {g}: {tpr:.3f}")
print(f"\nDiferencia de oportunidad igual: {diferencia:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(list(tprs_por_grupo.keys()), list(tprs_por_grupo.values()),
       color=["#4C72B0", "#DD8452"], width=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Tasa de verdaderos positivos (TPR)")
ax.set_title("Auditoría de equidad: TPR por grupo demográfico")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
def calcular_diferencia_equidad(y_real, y_predicho, grupo):
    """
    Calcola il tasso di veri positivi (TPR) di ciascun gruppo demografico
    e la differenza tra il gruppo con TPR migliore e quello con TPR peggiore
    (differenza di opportunità uguale).
    """
    grupos_unicos = np.unique(grupo)
    tprs = {}

    for g in grupos_unicos:
        mascara_grupo = grupo == g
        y_real_grupo = y_real[mascara_grupo]
        y_predicho_grupo = y_predicho[mascara_grupo]

        # TPR = proporzione di positivi reali che il modello predice anch'esso come positivi
        positivos_reales = y_real_grupo == 1
        if positivos_reales.sum() == 0:
            tpr_grupo = np.nan
        else:
            tpr_grupo = (y_predicho_grupo[positivos_reales] == 1).mean()

        tprs[g] = tpr_grupo

    diferencia = max(tprs.values()) - min(tprs.values())

    return tprs, diferencia


tprs_por_grupo, diferencia = calcular_diferencia_equidad(y_test, y_pred, grupo_test)

print("TPR por grupo:")
for g, tpr in tprs_por_grupo.items():
    print(f"  Grupo {g}: {tpr:.3f}")
print(f"Diferencia de oportunidad igual: {diferencia:.3f}")
```
</details>

## Esercizio 2: stima quale variabile pesa di più nelle decisioni del modello

Il modello che hai verificato nell'esercizio precedente decide a partire da tre variabili: età, reddito e anzianità lavorativa. Ma un modello di apprendimento automatico non ti dice di default quale variabile pesi di più in ogni decisione: è una scatola nera, come la consulenza medica senza spiegazione che hai visto nella Lezione 12. L'importanza per permutazione è una tecnica semplice di IA spiegabile (XAI) che apre quella scatola: se permuti — mescoli casualmente — i valori di una colonna e l'accuratezza del modello cala molto, quella colonna era importante per le sue decisioni; se l'accuratezza cambia appena, il modello la usa poco.

Completa la funzione `importancia_por_permutacion(modelo, X, y, columna)`, che segue questi passi:

1. Calcola l'accuratezza del modello con i dati originali
2. Crea una copia di `X` e permuta — mescola casualmente — solo i valori della colonna indicata, lasciando intatte le altre colonne
3. Calcola l'accuratezza del modello con quella colonna permutata
4. Restituisci il calo di accuratezza: accuratezza originale meno accuratezza con la colonna permutata

Riutilizza `modelo`, `X_test` e `y_test` dell'esercizio 1.

**Suggerimenti:**

- `accuracy_score(y_real, y_predicho)`, da `sklearn.metrics`, calcola l'accuratezza
- `rng.permutation(X_permutado[:, columna])` restituisce una versione mescolata di quella colonna, senza modificare il resto di `X_permutado`
- Più grande è il calo di accuratezza, più importante è la variabile per il modello

In [ ]:
from sklearn.metrics import accuracy_score


def importancia_por_permutacion(modelo, X, y, columna, semilla=42):
    """
    Estima la importancia de una columna de X para las predicciones de
    `modelo`, midiendo cuánto empeora la precisión cuando se permutan
    (barajan aleatoriamente) sus valores.

    Parámetros
    ----------
    modelo : modelo ya entrenado, con método .predict()
    X : matriz de variables (numpy array)
    y : etiquetas reales
    columna : índice de la columna de X que quieres evaluar
    semilla : semilla del generador aleatorio, para que el resultado sea reproducible

    Devuelve
    --------
    caida_precision : precisión original menos precisión con la columna permutada
    """
    rng = np.random.default_rng(semilla)

    y_pred_original = modelo.predict(X)
    precision_original = accuracy_score(y, y_pred_original)

    X_permutado = X.copy()
    # TODO: sustituye la columna `columna` de X_permutado por una versión
    # barajada aleatoriamente de sus propios valores, usando rng.permutation
    X_permutado[:, columna] = ...

    y_pred_permutado = modelo.predict(X_permutado)
    precision_permutada = accuracy_score(y, y_pred_permutado)

    # TODO: calcula la caída de precisión (precisión original menos precisión permutada)
    caida_precision = ...

    return caida_precision


importancias = {}
for i, nombre in enumerate(nombres_columnas):
    importancias[nombre] = importancia_por_permutacion(modelo, X_test, y_test, i)

print("Caída de precisión al permutar cada variable:")
for nombre, caida in importancias.items():
    print(f"  {nombre}: {caida:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(list(importancias.keys()), list(importancias.values()), color="#55A868", width=0.5)
ax.set_ylabel("Caída de precisión al permutar la variable")
ax.set_title("Importancia por permutación de cada variable")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Vedi soluzione</summary>

```python
from sklearn.metrics import accuracy_score


def importancia_por_permutacion(modelo, X, y, columna, semilla=42):
    """
    Stima l'importanza di una colonna di X per le previsioni di
    `modelo`, misurando quanto peggiora l'accuratezza quando se ne
    permutano (mescolano casualmente) i valori.
    """
    rng = np.random.default_rng(semilla)

    y_pred_original = modelo.predict(X)
    precision_original = accuracy_score(y, y_pred_original)

    X_permutado = X.copy()
    X_permutado[:, columna] = rng.permutation(X_permutado[:, columna])

    y_pred_permutado = modelo.predict(X_permutado)
    precision_permutada = accuracy_score(y, y_pred_permutado)

    caida_precision = precision_original - precision_permutada

    return caida_precision


importancias = {}
for i, nombre in enumerate(nombres_columnas):
    importancias[nombre] = importancia_por_permutacion(modelo, X_test, y_test, i)

print("Calo di accuratezza permutando ciascuna variabile:")
for nombre, caida in importancias.items():
    print(f"  {nombre}: {caida:.3f}")
```
</details>

## Per proseguire

Hai appena verificato un modello per equità e aperto la sua scatola nera con una tecnica di interpretabilità: due elementi centrali dell'IA responsabile che hai visto nella Lezione 12. Questo stesso schema — misurare prima di presumere, mettere in discussione prima di mettere in produzione — distingue un modello tecnicamente competente da un modello che è anche sicuro per le persone a cui si rivolge.

Se vuoi andare oltre, aggiungi una terza metrica di equità, la parità demografica, e confrontala con la differenza di opportunità uguale sugli stessi dati: raccontano la stessa storia o no?

**Applica questa verifica a un modello reale del tuo team, anche in modo informale.**